In [ ]:
# Imports and random seed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42

In [ ]:
# Evaluation function
def evaluate(name, model):
    model.fit(X_tr, y_train)
    pred = model.predict(X_va)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae  = mean_absolute_error(y_val, pred)
    r2   = r2_score(y_val, pred)
    print(f'{name:22} RMSE={rmse:6.2f}  MAE={mae:6.2f}  R2={r2:6.3f}')
    return model

In [ ]:
# Importing the data
df_train= pd.read_csv('Train.csv')
df_train.head()

In [ ]:
# Separation to train and val sets (what you sent me)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(df_train, groups=df_train['Place_ID']))

val_places = df_train.iloc[val_idx]['Place_ID'].unique()

place_split = df_train[['Place_ID']].drop_duplicates()
place_split['split'] = 'train'
place_split.loc[place_split['Place_ID'].isin(val_places), 'split'] = 'val'

place_split.to_csv('shared_place_split.csv', index=False)

place_split = pd.read_csv('shared_place_split.csv')
df = df_train.merge(place_split, on='Place_ID')

X_train = df[df.split == 'train'].drop(columns=['target', 'split'])
y_train = df[df.split == 'train']['target']
X_val   = df[df.split == 'val'].drop(columns=['target', 'split'])
y_val   = df[df.split == 'val']['target']

assert set(X_train['Place_ID']) & set(X_val['Place_ID']) == set()  # sanity check

print('X_train:', X_train.shape)
print('X_val:',X_val.shape)

In [ ]:
# Separation to Weather dataset
weather_cols = [
    'precipitable_water_entire_atmosphere',
    'relative_humidity_2m_above_ground',
    'specific_humidity_2m_above_ground',
    'temperature_2m_above_ground',
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
]
X_weather = X_train[weather_cols].copy()
print(X_weather.shape)
X_weather.head()

In [ ]:
# Weather data frame (with targets)
df_weather = pd.concat([X_weather, y_train], axis=1)
df_weather.head()

In [ ]:
# Correlation heat map
plt.figure(figsize=(7, 5))
sns.heatmap(df_weather.corr(),
            annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Weather features to target correlation map')
plt.xticks(rotation=45, ha='right')  
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Removing specific humidity, due to high correlation with precipitable_water_entire_atmosphere (included in it)
df_weather= df_weather.drop(columns='specific_humidity_2m_above_ground')
df_weather.columns

In [ ]:
# Feature engineering of wind
u = df_weather['u_component_of_wind_10m_above_ground']
v = df_weather['v_component_of_wind_10m_above_ground']

# Calculating wind speed and direction (angle)
df_weather['wind_speed']     = np.sqrt(u**2 + v**2)
df_weather['wind_direction'] = (np.degrees(np.arctan2(v, u)) + 360) % 360   # 0–360°
rad = np.radians(df_weather['wind_direction'])
df_weather['wind_dir_sin'] = np.sin(rad)
df_weather['wind_dir_cos'] = np.cos(rad)
# Why sin and cos: Wind direction is circular (0° = 360°), so we encode it as sin and cos 
# to remove the false jump at the wrap-around and let the model see that directions like 359° 
# and 1° are actually close.

# Wind speed and target corelation:
sns.regplot(x='wind_speed', y='target', data=df_weather, lowess=True, scatter_kws={'alpha':0.1})